# 8 — Sequential-slide integration

Four pairing levels, weakest assumptions first. Cell-to-cell pairing is **not** among them —
across serial sections the two modalities measure different cells, and no amount of
registration quality changes that.

| level | unit | needs registration |
|---|---|---|
| donor | `donor_id` | no |
| niche | kNN composition class | no |
| structure | matched islet | yes |
| pseudo-cell | inferred link | yes, QC-gated, labelled inference |

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from phenocycler import load_config

cfg = load_config(pathlib.Path.cwd().parent / 'config.ini')
print('mode:', cfg.integration_mode)

## S8 — donor level (no registration)

The floor of the whole integration, and it works even when registration fails.

Note the three fractions. PhenoCycler's `Epithelial` is a *default sink* — every cell that
fails all positive gates lands there, flagged `epi_default` — while Xenium's `Exocrine` is a
positive call. Comparing raw fractions measures that methodological difference, not biology,
so the headline number uses `frac_evidence_positive` on both sides.

In [ ]:
from phenocycler.integration.donor import run_donor

donor_stats = run_donor(cfg, roi='panc')
donor_stats

## S7 — niches

Mirrors the Xenium pipeline's own niche definition (kNN composition → KMeans → majority
smoothing) over the *harmonised* vocabulary, fitted jointly across both sections so a niche
id means the same thing in both. Needs no registration.

The hex-grid view underneath it does depend on registration, which is what makes it a
spatially-resolved check on the alignment.

In [ ]:
from phenocycler.integration.grid import run_grid

niche_stats = run_grid(cfg, roi='panc')
niche_stats

## S6 — matched islets (the primary sequential result)

Islets persist across a few microns of sectioning, there are tens to hundreds per section,
and they are the unit T1D biology is about. Matching is a one-to-one Hungarian assignment on
distance + area + shape, gated before the optimiser runs.

`z` compares the match count against a permuted null — a dense islet field always produces
*some* matches within 200 µm by chance.

In [ ]:
from phenocycler.integration.match import run_match

match_stats = run_match(cfg, roi='panc')
match_stats

In [ ]:
import pandas as pd

path = cfg.paired_dir / 'islets.parquet'
if path.exists():
    pairs = pd.read_parquet(path)
    print(f'{len(pairs)} matched islet pairs across {pairs.donor_id.nunique()} donor(s)')
    display(pairs.head(10))

## S9 — pseudo-cell links (inference, not measurement)

MaxFuse-class linking over the verified protein↔gene anchors. Runs within matched islets or
niches so the assignment stays small and spatially constrained.

This will **refuse** to run below `crossmodal_min_anchors`. That is not a nuisance: INS,
GCG, SST and Vimentin are off the Xenium panel, so the usable anchor set is much thinner
than "46 of 55 markers" suggests, and linking on a handful of features produces
confident-looking nonsense.

In [ ]:
from phenocycler.integration.crossmodal import run_crossmodal

cm = run_crossmodal(cfg, roi='panc')
cm

## Report and figures

Scientific acceptance, in order of how much each demands:

1. donor composition correlates across modalities;
2. the ND → AAB → T1D β-loss gradient appears in **both** modalities independently;
3. on matched islets, protein INS⁺ fraction tracks Xenium Beta fraction and insulitis grades
   agree;
4. permuted-pairing and rotation nulls destroy all of the above.

In [ ]:
from phenocycler.integration.qc import run_qc
from phenocycler.integration.figures import run_figures

run_qc(cfg, roi='panc')
paths = run_figures(cfg, roi='panc')

In [ ]:
from IPython.display import Image, display

for name in ('composition.png', 'disease_trend.png', 'islet_concordance.png'):
    p = cfg.integration_figures_dir / name
    if p.exists():
        print(name)
        display(Image(filename=str(p)))